In [2]:
import warnings
from pathlib import Path

import joblib
import keras
import numpy as np
from scipy.sparse import csr_matrix, hstack
from sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score

warnings.filterwarnings("ignore")
np.random.seed(42)
keras.utils.set_random_seed(42)

MODEL_DIR = Path("../models")

TARGETS = {
    "phishing_precision": 0.90,
    "phishing_recall": 0.85,
    "audio_auc": 0.85,
    "video_auc": 0.80,
}

results = {}
print("TDD Target Metrics:")
for k, v in TARGETS.items():
    print(f"  {k}: {v}")

# 1) Phishing evaluation
vec_path = MODEL_DIR / "phishing_tfidf_vectorizer.pkl"
clf_path = MODEL_DIR / "phishing_classifier.pkl"

if vec_path.exists() and clf_path.exists():
    vectorizer = joblib.load(vec_path)
    classifier = joblib.load(clf_path)

    test_texts = [
        "URGENT: Your SBI account will be suspended. Verify at http://sbi-fake.xyz/login now.",
        "Your account statement for January 2024 is ready. View at https://www.hdfcbank.com.",
        "Click here to claim your lottery prize of Rs. 10 Lakhs immediately!",
        "Meeting reminder: Sprint planning tomorrow at 10 AM IST.",
        "Your KYC is expiring. Share Aadhaar and OTP to update. http://kyc-update.xyz",
        "Your Flipkart order has been shipped. Track at https://www.flipkart.com/track.",
    ]
    test_labels = np.array([1, 0, 1, 0, 1, 0])

    x_test_text = vectorizer.transform(test_texts)

    n_total_features = getattr(classifier, "n_features_in_", x_test_text.shape[1])
    n_url_feats = max(0, n_total_features - x_test_text.shape[1])
    if n_url_feats > 0:
        x_test = hstack([x_test_text, csr_matrix(np.zeros((len(test_texts), n_url_feats)))])
    else:
        x_test = x_test_text

    y_pred = classifier.predict(x_test)
    y_prob = classifier.predict_proba(x_test)[:, 1]

    precision = precision_score(test_labels, y_pred)
    recall = recall_score(test_labels, y_pred)
    f1 = f1_score(test_labels, y_pred)

    results["phishing"] = {
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }

    print("\nPhishing Model Results:")
    print(f"  Precision: {precision:.4f} (target: {TARGETS['phishing_precision']})")
    print(f"  Recall:    {recall:.4f} (target: {TARGETS['phishing_recall']})")
else:
    print("\nPhishing model not found. Run notebook 01 first.")
    results["phishing"] = None


# 2) Audio evaluation
audio_path = MODEL_DIR / "audio_deepfake_cnn.h5"
if audio_path.exists():
    try:
        audio_model = keras.models.load_model(audio_path)
    except Exception as e:
        print(f"\nAudio model found but could not be loaded: {e}")
        results["audio"] = None
        audio_model = None

if audio_path.exists() and audio_model is not None:

    n_mels, spec_width = 128, 128

    def gen_audio_genuine(n):
        out = []
        for _ in range(n):
            base = np.random.uniform(0.1, 0.3, (n_mels, spec_width))
            for h in range(3, 8):
                fb = int(n_mels * h / 20)
                w = np.random.randint(3, 8)
                s, e = max(0, fb - w), min(n_mels, fb + w)
                base[s:e, :] += np.random.uniform(0.4, 0.8) * np.random.uniform(0.8, 1.0, (e - s, spec_width))
            out.append(np.clip(base, 0, 1))
        return np.array(out, dtype=np.float32)[..., np.newaxis]

    def gen_audio_deepfake(n):
        out = []
        for _ in range(n):
            base = np.random.uniform(0.1, 0.3, (n_mels, spec_width))
            for h in range(3, 8):
                fb = int(n_mels * h / 20)
                w = np.random.randint(2, 5)
                s, e = max(0, fb - w), min(n_mels, fb + w)
                base[s:e, :] += np.random.uniform(0.3, 0.7)
            artifact = 0.3 * np.sin(np.linspace(0, 12 * np.pi, spec_width))[np.newaxis, :]
            base[40:80, :] += artifact * np.random.uniform(0.5, 1.0)
            base[100:, :] *= 0.1
            out.append(np.clip(base, 0, 1))
        return np.array(out, dtype=np.float32)[..., np.newaxis]

    x_test_a = np.concatenate([gen_audio_genuine(30), gen_audio_deepfake(30)])
    y_test_a = np.concatenate([np.zeros(30), np.ones(30)])

    y_prob_a = audio_model.predict(x_test_a, verbose=0).flatten()
    auc_audio = roc_auc_score(y_test_a, y_prob_a)
    results["audio"] = {"auc": auc_audio}
    print(f"\nAudio Deepfake AUC-ROC: {auc_audio:.4f} (target: {TARGETS['audio_auc']})")
else:
    print("\nAudio model unavailable. Run notebook 02 first or re-export a valid model file.")
    results["audio"] = None


# 3) Video evaluation
video_path = MODEL_DIR / "video_deepfake_detector.h5"
if video_path.exists():
    try:
        video_model = keras.models.load_model(video_path)
    except Exception as e:
        print(f"\nVideo model found but could not be loaded: {e}")
        results["video"] = None
        video_model = None

if video_path.exists() and video_model is not None:
    frame_size = 128

    def gen_frames(n, deepfake=False):
        frames = []
        for _ in range(n):
            f = np.random.uniform(0.3, 0.7, (frame_size, frame_size, 3)).astype(np.float32)
            if deepfake:
                cy, cx = frame_size // 2, frame_size // 2
                y, x = np.ogrid[:frame_size, :frame_size]
                mask = ((y - cy) ** 2 + (x - cx) ** 2) > 40 ** 2
                f[mask] *= 0.9
                f[mask] += 0.05
                f[cy:cy + 3, :, :] += 0.1
            frames.append(np.clip(f, 0, 1))
        return np.array(frames, dtype=np.float32)

    x_test_v = np.concatenate([gen_frames(30, False), gen_frames(30, True)])
    y_test_v = np.concatenate([np.zeros(30), np.ones(30)])

    y_prob_v = video_model.predict(x_test_v, verbose=0).flatten()
    auc_video = roc_auc_score(y_test_v, y_prob_v)

    results["video"] = {"auc": auc_video}
    print(f"\nVideo Deepfake AUC-ROC: {auc_video:.4f} (target: {TARGETS['video_auc']})")
else:
    print("\nVideo model unavailable. Run notebook 03 first or re-export a valid model file.")
    results["video"] = None


# 4) Summary report
print("\n" + "=" * 60)
print("CyberShield AI — Model Evaluation Summary")
print("=" * 60)

report = []

if results.get("phishing"):
    p = results["phishing"]
    report.append(("Phishing Precision", p["precision"], TARGETS["phishing_precision"], p["precision"] >= TARGETS["phishing_precision"]))
    report.append(("Phishing Recall", p["recall"], TARGETS["phishing_recall"], p["recall"] >= TARGETS["phishing_recall"]))
else:
    report.append(("Phishing Precision", None, TARGETS["phishing_precision"], False))
    report.append(("Phishing Recall", None, TARGETS["phishing_recall"], False))

if results.get("audio"):
    report.append(("Audio AUC-ROC", results["audio"]["auc"], TARGETS["audio_auc"], results["audio"]["auc"] >= TARGETS["audio_auc"]))
else:
    report.append(("Audio AUC-ROC", None, TARGETS["audio_auc"], False))

if results.get("video"):
    report.append(("Video AUC-ROC", results["video"]["auc"], TARGETS["video_auc"], results["video"]["auc"] >= TARGETS["video_auc"]))
else:
    report.append(("Video AUC-ROC", None, TARGETS["video_auc"], False))

print(f"{'Metric':<25} {'Actual':>10} {'Target':>10} {'Status':>10}")
print("-" * 60)
for name, actual, target, passed in report:
    actual_str = f"{actual:.4f}" if actual is not None else "N/A"
    status = "PASS" if passed else ("FAIL" if actual is not None else "SKIP")
    print(f"{name:<25} {actual_str:>10} {target:>10.2f} {status:>10}")

passed_count = sum(1 for _, _, _, p in report if p)
print(f"\nOverall: {passed_count}/{len(report)} metrics meet targets")
print("Evaluation complete.")

TDD Target Metrics:
  phishing_precision: 0.9
  phishing_recall: 0.85
  audio_auc: 0.85
  video_auc: 0.8

Phishing Model Results:
  Precision: 0.0000 (target: 0.9)
  Recall:    0.0000 (target: 0.85)

Audio model found but could not be loaded: Unable to synchronously open file (file signature not found)

Audio model unavailable. Run notebook 02 first or re-export a valid model file.



Video Deepfake AUC-ROC: 1.0000 (target: 0.8)

CyberShield AI — Model Evaluation Summary
Metric                        Actual     Target     Status
------------------------------------------------------------
Phishing Precision            0.0000       0.90       FAIL
Phishing Recall               0.0000       0.85       FAIL
Audio AUC-ROC                    N/A       0.85       SKIP
Video AUC-ROC                 1.0000       0.80       PASS

Overall: 1/4 metrics meet targets
Evaluation complete.
